# CP1 Week 9 -- Functions: Building Blocks

**Course:** Computer Programming 1 (CP1) | **Session:** 5 hours

## Learning Objectives

1. Define functions with `def`, parameters, and `return`
2. Understand scope (local vs global variables)
3. Refactor messy code into clean, reusable functions
4. Build the pipeline function structure
5. Write functions that call other functions

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/ArifSolmaz/courseos-curriculum.git
# %cd courseos-curriculum/course-content

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: Function Anatomy

```python
def function_name(parameter1, parameter2):
    """Docstring: what this function does."""
    # body: the work
    result = parameter1 + parameter2
    return result     # send the answer back
```

```
  function_name  -- the name you call it by
  (parameters)   -- inputs the function needs
  docstring      -- description (for humans)
  body           -- the actual code
  return         -- the output
```


In [ ]:
def calculate_mean(values):
    """Calculate the arithmetic mean of a list of numbers."""
    if not values:
        return 0
    return sum(values) / len(values)

# Using the function
data = [10, 20, 30, 40, 50]
result = calculate_mean(data)
print(f"Mean of {data} = {result}")

# Functions can call other functions!
def summarize(values):
    """Return a summary dict."""
    return {
        "mean": calculate_mean(values),
        "min": min(values) if values else 0,
        "max": max(values) if values else 0,
        "count": len(values),
    }

print(f"Summary: {summarize(data)}")

**Expected Output:**
```
Mean of [10, 20, 30, 40, 50] = 30.0
Summary: {'mean': 30.0, 'min': 10, 'max': 50, 'count': 5}
```

### Example 2 -- Functions with multiple return values

In [ ]:
def analyze_values(values):
    """Return multiple statistics as a tuple."""
    if not values:
        return 0, 0, 0, 0

    mean = sum(values) / len(values)
    minimum = min(values)
    maximum = max(values)
    spread = maximum - minimum
    return mean, minimum, maximum, spread

# Unpack the results
data = [10, 25, 30, 15, 20, 35, 5, 40]
avg, lo, hi, spread = analyze_values(data)
print(f"Mean: {avg:.1f}")
print(f"Range: {lo} to {hi} (spread: {spread})")

**Expected Output:**
```
Mean: 22.5
Range: 5 to 40 (spread: 35)
```

### Example 3 -- Functions that return dictionaries

In [ ]:
def compute_stats(values):
    """Return stats as a dictionary for easy access."""
    if not values:
        return {"count": 0, "mean": 0, "min": 0, "max": 0}

    n = len(values)
    m = sum(values) / n
    return {
        "count": n,
        "mean": round(m, 2),
        "min": min(values),
        "max": max(values),
        "sum": sum(values),
    }

stats = compute_stats([10, 20, 30, 40, 50])
print("Stats:", stats)
print(f"The mean is {stats['mean']}")

**Expected Output:**
```
Stats: {'count': 5, 'mean': 30.0, 'min': 10, 'max': 50, 'sum': 150}
The mean is 30.0
```

Returning a dictionary is often better than a tuple because you access values
by name (`stats["mean"]`) instead of position, which is less error-prone.

---
## Part 2: Refactoring -- Before & After

**Refactoring** means restructuring code without changing what it does.
The goal: make it clearer, shorter, and reusable.

In [ ]:
# BEFORE: one messy block
raw = [{"val": "25"}, {"val": ""}, {"val": "abc"}, {"val": "50"}, {"val": "-10"}]
cleaned = []
for row in raw:
    v = row["val"]
    if v == "":
        continue
    try:
        num = float(v)
    except ValueError:
        continue
    if num < 0:
        continue
    cleaned.append(num)
mean_val = sum(cleaned) / len(cleaned) if cleaned else 0
print(f"BEFORE result: {mean_val}")

# AFTER: organized into functions
def parse_value(text):
    """Convert text to float, return None if invalid."""
    if not text or text.strip() == "":
        return None
    try:
        return float(text)
    except ValueError:
        return None

def is_in_range(value, low=0, high=1000):
    """Check if value is within range."""
    return low <= value <= high

def clean_values(raw_data):
    """Clean raw dicts, return list of floats."""
    result = []
    for row in raw_data:
        val = parse_value(row.get("val", ""))
        if val is not None and is_in_range(val):
            result.append(val)
    return result

cleaned2 = clean_values(raw)
mean_val2 = calculate_mean(cleaned2)
print(f"AFTER result:  {mean_val2}")

**Expected Output:**
```
BEFORE result: 37.5
AFTER result:  37.5
```

Same result, but the AFTER version is:
- **Readable** -- you can understand it without tracing every line
- **Testable** -- you can test `parse_value()` independently
- **Reusable** -- `is_in_range()` works on any number

---
## Part 3: Your Pipeline Functions

In [ ]:
def load_data(config):
    """Load raw data from source."""
    data = []
    for i in range(config.get("n_points", 10)):
        data.append({"index": i, "value": str(20 + i * 3.5)})
    print(f"Loaded {len(data)} rows")
    return data

def clean_data(data, config):
    """Clean data: parse values, filter invalid."""
    cleaned = []
    for row in data:
        val = parse_value(row.get("value", ""))
        if val is not None and is_in_range(val, config.get("min", 0), config.get("max", 100)):
            cleaned.append({**row, "value": val})
    print(f"Cleaned: {len(data)} -> {len(cleaned)}")
    return cleaned

def analyze(clean_data_list, config):
    """Analyze data: compute stats."""
    values = [r["value"] for r in clean_data_list]
    stats = summarize(values)
    print(f"Analysis: mean={stats['mean']:.2f}")
    return {"analysis_summary": stats}

# Run the pipeline
config = {"n_points": 20, "min": 0, "max": 80}
data = load_data(config)
cleaned = clean_data(data, config)
results = analyze(cleaned, config)
print(f"Results: {results}")

### Understanding Scope

In [ ]:
# Variables inside a function are LOCAL -- they only exist inside.
# Variables outside are GLOBAL.

message = "I am global"

def my_function():
    message = "I am local"   # this is a DIFFERENT variable!
    secret = 42              # only exists inside the function
    print(f"  Inside: message = '{message}'")
    print(f"  Inside: secret = {secret}")

my_function()
print(f"  Outside: message = '{message}'")
# print(f"  Outside: secret = {secret}")  # ERROR! secret does not exist here

**Expected Output:**
```
  Inside: message = 'I am local'
  Inside: secret = 42
  Outside: message = 'I am global'
```

The global `message` is unchanged because the function created its own local copy.

### Common Mistakes with Functions

| Mistake | Example | Fix |
|---------|---------|-----|
| Forgetting return | `def add(a,b): a+b` | `def add(a,b): return a+b` |
| Calling without () | `result = my_func` | `result = my_func()` |
| Wrong argument count | `add(1)` when def is `add(a,b)` | Pass all required args |
| Modifying global state | Using global variables inside functions | Pass values as parameters |

### Debugging Tip

If your function returns `None` unexpectedly, you probably forgot the `return`
statement. Without `return`, Python automatically returns `None`.

### Try It Yourself

In [ ]:
# TODO: Refactor this code into 3 functions:
# 1. parse_values(raw) -> list of floats (skip non-numeric)
# 2. filter_range(values, lo, hi) -> list within range
# 3. compute_mean(values) -> float

raw_data = ["25", "abc", "50", "-10", "75", "", "100"]

# Messy version (refactor this!):
clean = []
for item in raw_data:
    try:
        v = float(item)
        if 0 <= v <= 100:
            clean.append(v)
    except ValueError:
        pass
if clean:
    avg = sum(clean) / len(clean)
    print(f"Average: {avg:.2f}")

---
## Key Takeaways -- Week 9

1. **Functions** encapsulate reusable logic: `def name(params): ... return`
2. **Refactoring** improves structure without changing behavior
3. **Small functions** are easier to test, debug, and reuse
4. **Scope** -- local variables stay inside functions
5. **Pipeline functions** should be independent and composable

---
## Homework

### Review (R1-R4)

In [ ]:
# R1: What are the 3 parts of a function definition?
# R2: What is the difference between parameters and arguments?
# R3: What does return do?
# R4: What is refactoring?

### Practice (P1-P5)

In [ ]:
# P1: Write a function that removes duplicates from a list.


In [ ]:
# P2: Write 3 helper functions and one main that calls all 3.


In [ ]:
# P3: Refactor this into clean functions:
data = [10, 20, 30, 40, 50]
total = 0
for x in data:
    total += x
mean = total / len(data)
dev = 0
for x in data:
    dev += (x - mean) ** 2
std = (dev / len(data)) ** 0.5
print(f"Mean: {mean}, Std: {std}")


In [ ]:
# P4: Write a complete mini-pipeline:
# load_data() -> clean_data() -> analyze() -> print_report()


In [ ]:
# P5: Write a function that accepts another function as a parameter.
# Example: apply_to_all(data, transform_func)


### Challenge (C1-C3)

In [ ]:
# C1: Write a function that returns multiple values as a tuple.


In [ ]:
# C2: Write a "pipeline runner" that takes a list of functions and data,
# and applies each function in sequence.


In [ ]:
# C3: Write tests for each of your pipeline functions.


### Mini-Project

In [ ]:
# M1: Refactoring Challenge
# Take the messy code below and refactor it into 5+ clean functions.
# The output must be identical before and after.

data = ["25", "abc", "", "50", "-10", "30", "999", "42"]
results = []
dropped = 0
for item in data:
    if item == "":
        dropped += 1
        continue
    try:
        val = float(item)
    except ValueError:
        dropped += 1
        continue
    if val < 0 or val > 100:
        dropped += 1
        continue
    results.append(val)
if results:
    avg = sum(results) / len(results)
    mn = min(results)
    mx = max(results)
    print(f"Processed {len(results)} values (dropped {dropped})")
    print(f"Mean: {avg:.2f}, Min: {mn}, Max: {mx}")


---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)